In [3]:
!pip install transformers jiwer librosa soundfile torch accelerate

In [5]:
import os
import torch
from transformers import pipeline
from jiwer import wer

# 1. Khởi tạo phần cứng và mô hình
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Đang sử dụng phần cứng: {device}")

# Sử dụng Whisper bản 'small' để cân bằng giữa tốc độ và độ chính xác
pipe = pipeline(
    "automatic-speech-recognition",
    model="vinai/PhoWhisper-small", # Hoặc "vinai/PhoWhisper-base"
    device=device,
    chunk_length_s=30
)
# 2. Cấu hình đường dẫn dữ liệu (THAY ĐỔI ĐƯỜNG DẪN NÀY)
# Dán đường dẫn bạn copy được từ Kaggle vào đây
base_path = "/kaggle/input/datasets/tuannguyenvananh/vivos-dataset/vivos" 
test_path = os.path.join(base_path, "test")
prompts_file = os.path.join(test_path, "prompts.txt")
waves_dir = os.path.join(test_path, "waves")

# 3. Đọc dữ liệu chuẩn (Ground Truth)
ground_truths = {}
with open(prompts_file, 'r', encoding='utf-8') as f:
    for line in f:
        # File có định dạng: VIVOSDEV01_001 nội dung văn bản
        parts = line.strip().split(' ', 1)
        if len(parts) == 2:
            audio_id, transcript = parts
            ground_truths[audio_id] = transcript.lower()

import librosa # Thêm thư viện đọc audio

# 4. Quá trình kiểm thử (Chạy thử 10 mẫu trước)
predictions = []
references = []
count = 0
max_test = 10 

import re

# Thêm hàm xóa dấu câu
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text) # Xóa các ký tự không phải chữ/số/khoảng trắng
    return text

print("Bắt đầu nhận diện...")
for speaker in os.listdir(waves_dir):
    speaker_dir = os.path.join(waves_dir, speaker)
    if not os.path.isdir(speaker_dir):
        continue

    for audio_file in os.listdir(speaker_dir):
        if audio_file.endswith(".wav"):
            audio_id = audio_file.replace(".wav", "")
            
            if audio_id not in ground_truths:
                continue
                
            audio_path = os.path.join(speaker_dir, audio_file)
            
            # --- PHẦN CODE ĐƯỢC FIX ---
            # Đọc file bằng librosa và đưa về tần số lấy mẫu 16kHz (bắt buộc với Whisper)
            speech, sample_rate = librosa.load(audio_path, sr=16000)
            
            
            result = pipe(speech, generate_kwargs={"language": "vietnamese"})
            pred_text = clean_text(result["text"])
            
            predictions.append(pred_text)
            references.append(ground_truths[audio_id])
            
            print(f"[{count+1}] Audio ID: {audio_id}")
            print(f"Chuẩn: {ground_truths[audio_id]}")
            print(f"AI   : {pred_text}\n")
            
            count += 1
            if max_test and count >= max_test:
                break
    if max_test and count >= max_test:
        break

# 5. Đánh giá % lỗi (WER)
if len(predictions) > 0:
    error_rate = wer(references, predictions)
    print(f"==> Tỷ lệ lỗi (WER) trên {count} mẫu: {error_rate * 100:.2f}%")
else:
    print("Lỗi: Không tìm thấy file dữ liệu nào hợp lệ!")

Đang sử dụng phần cứng: cpu


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/vinai/PhoWhisper-small/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.p

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Bắt đầu nhận diện...
[1] Audio ID: VIVOSDEV19_196
Chuẩn: người anh sau đó được phép thầu xây kênh này nhưng không tiến hành
AI   : người anh sau đó được phép thầu xây kênh này nhưng không tiến hành

[2] Audio ID: VIVOSDEV19_020
Chuẩn: lên các cuộc hẹn phỏng vấn và thực hiện phỏng vấn
AI   : lên các cuộc hẹn phỏng vấn và thực hiện phỏng vấn

[3] Audio ID: VIVOSDEV19_030
Chuẩn: phù sa bồi đắp ruộng vườn cho xóm làng trù phú
AI   : phù sa bồi đắp ruộng vườn cho xóm làng trù phú

[4] Audio ID: VIVOSDEV19_262
Chuẩn: để rồi chúng ta khao khát mãi trong vòng luân hồi nhân gian
AI   : để rồi chúng ta khao khát mãi trong vòng luân hồi nhân gian

[5] Audio ID: VIVOSDEV19_064
Chuẩn: chóng mặt còn có thể do tình trạng rối loạn nhịp tim nặng gây ra
AI   : chóng mặt còn có thể do tình trạng rối loạn nhịp tim nặng gây ra

[6] Audio ID: VIVOSDEV19_056
Chuẩn: về nguyên tắc vào học là sinh viên phải đóng học phí
AI   : về nguyên tắc vào học là sinh viên phải đóng học phí

[7] Audio ID: VIVOSDEV19_218
Ch